In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP — Clone Repo & Install Packages
# ═══════════════════════════════════════════════════════════════

import os

# Clone repo
!git clone https://github.com/aneek22112007-tech/SocraticRL
%cd SocraticRL

print("✅ Repo cloned")

# Install packages
!pip install -q transformers torch peft bitsandbytes wandb datasets

print("✅ Packages installed")

# Login to HuggingFace
from huggingface_hub import login
login()

print("✅ HuggingFace login complete")

# Login to W&B
import wandb
wandb.login()

print("\n🎉 Setup complete!")


fatal: destination path 'SocraticRL' already exists and is not an empty directory.
/content/SocraticRL
✅ Repo cloned
✅ Packages installed


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ HuggingFace login complete


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: das-aneek007 (das-aneek007-polaris-school-of-teachnology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



🎉 Setup complete!


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("Loading Qwen2.5-7B in 4-bit...")

# 🔥 FORCE FP16 (IMPORTANT FIX)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # already correct
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,   # 🔥 ADD THIS LINE (VERY IMPORTANT)
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# 🔥 ALSO ADD THIS (prevents padding issues later)
tokenizer.pad_token = tokenizer.eos_token

print("✅ Model loaded in 4-bit")

# Add LoRA adapters
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)

# 🔥 ADD THIS (important for training stability)
model.config.use_cache = False

print("✅ LoRA adapters added")


Loading Qwen2.5-7B in 4-bit...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Model loaded in 4-bit
✅ LoRA adapters added


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Sanity Check — Verify Reward Function
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, '/content/SocraticRL')

from reward import compute_reward

print("Testing reward function...\n")

good = 'If you dropped a feather and a hammer in a vacuum, what would you expect to happen?'
bad  = 'The answer is that heavier objects fall faster due to gravity.'

progress_keywords = ['acceleration', 'same rate', 'air resistance', 'galileo', 'mass', 'vacuum']

r_good = compute_reward(good, 'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])
r_bad  = compute_reward(bad,  'falling objects', progress_keywords, understanding_score=0.1, turn_count=1, question_history=[])

print(f'Good question reward: {r_good.reward:+.2f}  ({r_good.feedback})')
print(f'Bad  answer  reward:  {r_bad.reward:+.2f}  ({r_bad.feedback})')

assert r_good.reward > 0, 'Good question should get positive reward'
assert r_bad.reward  < 0, 'Direct answer should get negative reward'

print('\n✅ Reward function OK')


Testing reward function...

Good question reward: +0.20  (is_question:+0.20)
Bad  answer  reward:  -0.80  (not_question:-0.30|direct_answer:-0.50)

✅ Reward function OK


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Build IMPROVED Training Dataset (50 scenarios × 3 variations)
# ═══════════════════════════════════════════════════════════════

from datasets import Dataset
from students.scenarios_expanded import TRAINING_SCENARIOS_EXPANDED

print(f"Building dataset from {len(TRAINING_SCENARIOS_EXPANDED)} scenarios...")

rows = []

for scenario in TRAINING_SCENARIOS_EXPANDED:
    # Variation 1: Direct Socratic prompt
    prompt1 = (
        f"""You are a Socratic tutor helping a student understand {scenario.topic}.

Student's current belief: {scenario.misconception}

Your role:
- Ask questions that challenge their thinking
- Never state the answer directly
- Guide them to discover the correct understanding themselves
- Be specific to the topic, not generic

What is your first Socratic question?"""
    )
    rows.append({"prompt": prompt1})

    # Variation 2: Misconception-focused
    prompt2 = (
        f"""Topic: {scenario.topic}
Student misconception: {scenario.misconception}

As a Socratic tutor, ask a question that:
1. Challenges this specific misconception
2. Doesn't give away the answer
3. Helps them think deeper

Your question:"""
    )
    rows.append({"prompt": prompt2})

    # Variation 3: Persona-based
    prompt3 = (
        f"""You are tutoring a {scenario.persona}.
They believe: {scenario.misconception}
Topic: {scenario.topic}

Ask a Socratic question that helps them reconsider their belief.
Question:"""
    )
    rows.append({"prompt": prompt3})

dataset = Dataset.from_list(rows)
print(f"✅ Dataset size: {len(dataset)} prompts")
print(f"   ({len(TRAINING_SCENARIOS_EXPANDED)} scenarios × 3 variations)")
print(f"\nExample prompt:\n{dataset[0]['prompt'][:300]}...")


Building dataset from 55 scenarios...
✅ Dataset size: 165 prompts
   (55 scenarios × 3 variations)

Example prompt:
You are a Socratic tutor helping a student understand Falling objects and gravity.

Student's current belief: Heavier objects fall faster than lighter objects.

Your role:
- Ask questions that challenge their thinking
- Never state the answer directly
- Guide them to discover the correct understandi...


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Define Reward Function for GRPO
# ═══════════════════════════════════════════════════════════════

import wandb
from reward import compute_reward
from students.scenarios_expanded import TRAINING_SCENARIOS_EXPANDED

# Initialize W&B
wandb.init(project='socratic-rl-training', resume='allow')
print(f"✅ W&B initialized: {wandb.run.url}")

def socratic_reward(prompts, completions, **kwargs):
    """
    GRPO reward function.
    prompts:     list of prompt strings
    completions: list of generated strings
    Returns:     list of float rewards
    """
    rewards = []

    for i, (prompt, completion) in enumerate(zip(prompts, completions)):
        # Extract topic from prompt
        topic = ''
        for line in prompt.split('\\n'):
            if 'Topic:' in line or 'topic' in line.lower():
                topic = line.split(':', 1)[-1].strip() if ':' in line else line
                break

        scenario_idx = (i // 3) % len(TRAINING_SCENARIOS_EXPANDED)
        scenario = TRAINING_SCENARIOS_EXPANDED[scenario_idx]
        question = completion.strip().splitlines()[0].strip() if completion.strip() else ""

        if not question or len(question) < 5:
            rewards.append(-1.0)
            continue

        # Score with reward.py
        progress_keywords = scenario.correct_answer.lower().split()
        result = compute_reward(
            question=question,
            topic=topic or scenario.topic,
            progress_keywords=progress_keywords,
            understanding_score=0.1,
            turn_count=1,
            question_history=[],
        )
        total_reward = result.reward
        rewards.append(total_reward)

    mean_reward = sum(rewards) / max(len(rewards), 1)
    wandb.log({'mean_reward': mean_reward})
    print(f'  Batch: mean_reward={mean_reward:.3f}')
    return rewards

print("✅ Reward function defined")


✅ W&B initialized: https://wandb.ai/das-aneek007-polaris-school-of-teachnology/socratic-rl-training/runs/v42aayeh
✅ Reward function defined


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: TRAIN — GRPO for 3 Epochs (IMPROVED)
# ═══════════════════════════════════════════════════════════════
!pip install -q trl
from trl import GRPOConfig, GRPOTrainer

print("Configuring GRPO trainer...")

training_args = GRPOConfig(
    output_dir='./socratic-agent',
    num_generations=2,
    max_completion_length=256,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_steps=50,
    report_to='wandb',
    fp16=False,
    bf16=False,   # 🔥 IMPORTANT FIX
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[socratic_reward],
    args=training_args,
    train_dataset=dataset.select(range(80)),
)

print("\n🚀 Starting GRPO training...")
print(f"Dataset size: {len(dataset)} samples")
print(f"Epochs: 1")
print(f"Watch for mean_reward to climb from ~-1.5 to ~+1.0\n")

trainer.train()

print("\n✅ Training complete!")


Configuring GRPO trainer...

🚀 Starting GRPO training...
Dataset size: 165 samples
Epochs: 1
Watch for mean_reward to climb from ~-1.5 to ~+1.0

  Batch: mean_reward=-0.025


Step,Training Loss
5,-0.000000
10,0.000000
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000


  Batch: mean_reward=0.225
  Batch: mean_reward=0.000
  Batch: mean_reward=0.250
  Batch: mean_reward=0.100
  Batch: mean_reward=-0.150
  Batch: mean_reward=0.025
  Batch: mean_reward=0.025
  Batch: mean_reward=0.425
  Batch: mean_reward=0.075
  Batch: mean_reward=0.100
  Batch: mean_reward=0.150
  Batch: mean_reward=-0.050
  Batch: mean_reward=0.075
  Batch: mean_reward=0.025
  Batch: mean_reward=0.050
  Batch: mean_reward=-0.025
  Batch: mean_reward=-0.075
  Batch: mean_reward=-0.050
  Batch: mean_reward=0.225
  Batch: mean_reward=0.225
  Batch: mean_reward=0.125
  Batch: mean_reward=0.075
  Batch: mean_reward=0.125
  Batch: mean_reward=0.075
  Batch: mean_reward=0.075
  Batch: mean_reward=0.100
  Batch: mean_reward=0.050
  Batch: mean_reward=-0.150
  Batch: mean_reward=0.225
  Batch: mean_reward=0.100
  Batch: mean_reward=0.300
  Batch: mean_reward=0.125
  Batch: mean_reward=0.100
  Batch: mean_reward=-0.075
  Batch: mean_reward=-0.050
  Batch: mean_reward=0.050
  Batch: mean_reward

Step,Training Loss
5,-0.000000
10,0.000000
15,0.000000
20,0.000000
25,0.000000
30,0.000000
35,0.000000
40,-0.000000



✅ Training complete!


In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Save Model & Push to Hub

print("Saving model...")
model.save_pretrained('socratic-agent-final')
tokenizer.save_pretrained('socratic-agent-final')
print("✅ Saved locally")

# 🔥 YOUR ACCOUNT
HF_USERNAME = "codepd"
repo_id = f"{HF_USERNAME}/socratic-tutor-v2"

print(f"\nPushing to HuggingFace Hub: {repo_id}")

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"✅ Pushed to https://huggingface.co/{repo_id}")

print("\n🎉 DONE!")


Saving model...
✅ Saved locally

Pushing to HuggingFace Hub: codepd/socratic-tutor-v2


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 26.1kB / 20.2MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp3ed99j7b/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

✅ Pushed to https://huggingface.co/codepd/socratic-tutor-v2

🎉 DONE!


AttributeError: 'Run' object has no attribute 'history'